In [1]:
"""
Problem: n factories, each with a different origin, towards multiple consumers.
Solution: 1 Dijkstra per factory (not per consumer).

Complexity:
    - 1 Dijkstra (min-heap):           O(E log V)
    - n factories (sequential CPU):    O(n * E log V)   -> linear in n
    - n factories (parallel, ideal):   O(E log V)       -> time of ONE run,
                                                             if n "workers" are free

Why it works without coordination between orders:
    Each factory is an independent source. A Dijkstra from factory F
    does not modify the graph or the state of another Dijkstra from factory G.
    There are no capacities being consumed, no supply/demand being deducted.
    Therefore it IS "embarrassingly parallel": you can run them all at once
    without any kind of lock, without updating anything shared.
"""

import heapq
import time
import random
from concurrent.futures import (
    ThreadPoolExecutor, ProcessPoolExecutor, as_completed
)
from dataclasses import dataclass, field


# ---------------------------------------------------------------------------
# 1. Graph representation
# ---------------------------------------------------------------------------

@dataclass
class Graph:
    """Weighted directed graph, represented as an adjacency list."""
    n_nodes: int
    adjacency: dict = field(default_factory=dict)  # node -> [(neighbor, weight), ...]

    def add_edge(self, u: int, v: int, weight: float,
                 bidirectional: bool = True):
        self.adjacency.setdefault(u, []).append((v, weight))
        if bidirectional:
            self.adjacency.setdefault(v, []).append((u, weight))

    def neighbors(self, u: int):
        return self.adjacency.get(u, [])


# ---------------------------------------------------------------------------
# 2. Individual Dijkstra (one source -> all nodes)
# ---------------------------------------------------------------------------

def dijkstra(graph: Graph, source: int):
    """
    Complexity: O(E log V) using a binary heap.

    Returns:
        dist:     dict node -> minimum distance from `source`
        previous: dict node -> previous node on the shortest path
                  (used to reconstruct the complete route)
    """
    dist = {source: 0.0}
    previous = {}
    visited = set()

    # heap of (accumulated_distance, node)
    heap = [(0.0, source)]

    while heap:
        current_distance, u = heapq.heappop(heap)

        if u in visited:
            continue
        visited.add(u)

        for v, weight in graph.neighbors(u):
            if v in visited:
                continue
            new_distance = current_distance + weight
            if new_distance < dist.get(v, float("inf")):
                dist[v] = new_distance
                previous[v] = u
                heapq.heappush(heap, (new_distance, v))

    return dist, previous


def reconstruct_path(previous: dict, source: int, destination: int):
    """Reconstructs the source -> destination path from the `previous` dict."""
    if destination != source and destination not in previous:
        return None  # unreachable

    path = [destination]
    while path[-1] != source:
        path.append(previous[path[-1]])
    path.reverse()
    return path


# ---------------------------------------------------------------------------
# 3. Solve the complete problem: n factories -> m consumers
# ---------------------------------------------------------------------------

def solve_sequential(graph: Graph,
                     factories: list[int],
                     consumers: list[int]):
    """
    Sequential CPU version: one Dijkstra per factory, one after another.
    Total complexity: O(n * E log V), linear in n = len(factories).
    """
    results = {}

    for factory in factories:
        dist, previous = dijkstra(graph, factory)

        factory_routes = {}
        for consumer in consumers:
            cost = dist.get(consumer, float("inf"))
            path = reconstruct_path(previous, factory, consumer)
            factory_routes[consumer] = {"cost": cost, "path": path}

        results[factory] = factory_routes

    return results


def solve_parallel(graph: Graph, factories: list[int], consumers: list[int],
                   max_workers: int = None):
    """
    Parallel version (simulates the "GPU" model: many Dijkstras at once).
    In real Python this uses threads + GIL, so effective parallelism
    for CPU-bound workloads is limited (see note below). The PATTERN is
    identical to what you would use in CUDA: one independent Dijkstra
    per "thread/block".

    Theoretical complexity: O(E log V) wall-clock time if there are
    min(n, available_workers) free compute units.
    """
    results = {}

    def task(factory):
        dist, previous = dijkstra(graph, factory)
        factory_routes = {}
        for consumer in consumers:
            cost = dist.get(consumer, float("inf"))
            path = reconstruct_path(previous, factory, consumer)
            factory_routes[consumer] = {"cost": cost, "path": path}
        return factory, factory_routes

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(task, f): f for f in factories}
        for future in as_completed(futures):
            factory, factory_routes = future.result()
            results[factory] = factory_routes

    return results


def solve_parallel_processes(graph: Graph,
                             factories: list[int],
                             consumers: list[int],
                             max_workers: int = None):
    """
    REAL parallelism for CPU-bound workloads: separate processes, each with
    its own Python interpreter (without a shared GIL). This is the correct
    "GPU" analogue within the limitations of pure Python.

    Note: each process receives a copy of the graph (pickling), so there is
    serialization overhead. For large graphs and large n, this cost is
    amortized; for small graphs, it may not be worthwhile (just like
    launching a GPU kernel for a trivial task).
    """
    results = {}

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_process_task, graph, factory, consumers): factory
            for factory in factories
        }
        for future in as_completed(futures):
            factory, factory_routes = future.result()
            results[factory] = factory_routes

    return results


def _process_task(graph, factory, consumers):
    """Module-level function (required so ProcessPoolExecutor can pickle the task)."""
    dist, previous = dijkstra(graph, factory)
    factory_routes = {}
    for consumer in consumers:
        cost = dist.get(consumer, float("inf"))
        path = reconstruct_path(previous, factory, consumer)
        factory_routes[consumer] = {"cost": cost, "path": path}
    return factory, factory_routes


# ---------------------------------------------------------------------------
# 4. Demo: compare sequential vs parallel times
# ---------------------------------------------------------------------------

def generate_random_graph(n_nodes: int, n_edges: int,
                          max_weight: float = 20.0, seed: int = 42):
    random.seed(seed)
    graph = Graph(n_nodes=n_nodes)
    for _ in range(n_edges):
        u = random.randint(0, n_nodes - 1)
        v = random.randint(0, n_nodes - 1)
        if u != v:
            weight = round(random.uniform(1.0, max_weight), 2)
            graph.add_edge(u, v, weight)
    return graph


if __name__ == "__main__":
    # --- Demo parameters ---
    N_NODES = 8000         # size of the path graph (V)
    N_EDGES = 50000        # (E)
    N_FACTORIES = 60       # n
    N_CONSUMERS = 30       # m

    graph = generate_random_graph(N_NODES, N_EDGES)
    factories = random.sample(range(N_NODES), N_FACTORIES)
    consumers = random.sample(range(N_NODES), N_CONSUMERS)

    print(f"Graph: V={N_NODES}, E={N_EDGES}")
    print(f"Factories (n): {N_FACTORIES}   Consumers (m): {N_CONSUMERS}")
    print("-" * 60)

    # --- Sequential ---
    t0 = time.perf_counter()
    res_seq = solve_sequential(graph, factories, consumers)
    t1 = time.perf_counter()
    print(f"Sequential (CPU, one by one):     {t1 - t0:.4f} s")

    # --- Parallel (threads, limited by the GIL) ---
    t0 = time.perf_counter()
    res_par = solve_parallel(graph, factories, consumers, max_workers=8)
    t1 = time.perf_counter()
    print(f"Parallel with THREADS (limited by GIL):  {t1 - t0:.4f} s")

    # --- Parallel (processes, real parallelism) ---
    t0 = time.perf_counter()
    res_par_proc = solve_parallel_processes(
        graph, factories, consumers, max_workers=8
    )
    t1 = time.perf_counter()
    print(f"Parallel with PROCESSES (real):            {t1 - t0:.4f} s")

    # --- Verify that all versions produce the same result ---
    identical = True
    for f in factories:
        for c in consumers:
            cost_seq = res_seq[f][c]["cost"]
            cost_threads = res_par[f][c]["cost"]
            cost_processes = res_par_proc[f][c]["cost"]
            if not (cost_seq == cost_threads == cost_processes):
                identical = False
    print(f"\nIdentical results in all 3 versions? {identical}")

    # --- Show an example route ---
    example_factory = factories[0]
    example_consumer = consumers[0]
    info = res_seq[example_factory][example_consumer]
    print(f"\nExample -> Factory {example_factory} to Consumer {example_consumer}:")
    print(f"  Cost:  {info['cost']}")
    print(f"  Path:  {info['path']}")

Graph: V=8000, E=50000
Factories (n): 60   Consumers (m): 30
------------------------------------------------------------
Sequential (CPU, one by one):     3.1863 s
Parallel with THREADS (limited by GIL):  3.7025 s
Parallel with PROCESSES (real):            2.3466 s

Identical results in all 3 versions? True

Example -> Factory 401 to Consumer 1141:
  Cost:  25.86
  Path:  [401, 3828, 5315, 6850, 2697, 3665, 2731, 4576, 1141]
